Notebook de pruebas:
Tiene todas las funcionalidades del standalone (main_despliegue_standalone)

Nota: Si este notebook da errores de importación asociados a Cartopy (CRSS); se debe reiniciar el kernel y volver a ejecutar todo

In [1]:
####################### N O  T O C A R ############################################
%reload_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd

root_path = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), ".."))
sys.path.append(root_path)


# Importar configuraciones del modulo
from procesado_datos.config_modulo.config_procesado import config_modulo_procesado
# Importar configuraciones del submodulo de despliegue
from procesado_datos.transmision.configs.configuracion_transmision import config_transmision

# Importar manager de configuraciones
from procesado_datos.config_modulo.ProcesadoConfig import ProcesadoConfig

# Importar servicios necesarios
import procesado_datos.services.Carga.cargar_datos_csv as carga
from procesado_datos.services.Utils.utilidades import *
from procesado_datos.services.Correctores.corrector_utils import *
from procesado_datos.services.Graficado.graficar_series_y_guardar import graficar_series_y_guardar
from procesado_datos.services.Graficado.graficar_mapa_de_posiciones import graficar_mapa_de_posiciones

##################################################################################

In [2]:
# Crear la instancia del manager de configuraciones
config = ProcesadoConfig.from_sources(
    config_modulo_procesado,
    config_transmision,
)    

In [3]:
config.variables_a_graficar

['temperatura_mar',
 'u_corriente',
 'v_corriente',
 'rap_corriente',
 'dir_corriente',
 'voltaje']

In [4]:
rutas_de_sondas, seriales_encontrados = carga.buscar_nombre_de_archivo_de_sonda(config)
rutas_de_sondas

['C:\\Users\\Atmosfera\\Desktop\\datos_crudos\\doris\\todos_los_datos\\datos_Localizacion_4912208_TOTAL.csv',
 'C:\\Users\\Atmosfera\\Desktop\\datos_crudos\\doris\\todos_los_datos\\datos_Localizacion_4912223_TOTAL.csv',
 'C:\\Users\\Atmosfera\\Desktop\\datos_crudos\\doris\\todos_los_datos\\datos_Localizacion_9909282_TOTAL.csv',
 'C:\\Users\\Atmosfera\\Desktop\\datos_crudos\\doris\\todos_los_datos\\datos_Localizacion_4912205_TOTAL.csv',
 'C:\\Users\\Atmosfera\\Desktop\\datos_crudos\\doris\\todos_los_datos\\datos_Localizacion_4912209_TOTAL.csv']

In [5]:
diccionario_de_datos_de_sondas = carga.cargar_datos_de_sonda(rutas_de_sondas, seriales_encontrados)

Datos cargados correctamente para la sonda: 4912208
Datos cargados correctamente para la sonda: 4912223
Datos cargados correctamente para la sonda: 9909282
Datos cargados correctamente para la sonda: 4912205
Datos cargados correctamente para la sonda: 4912209


In [6]:
diccionario_de_datos_de_sondas.keys()

dict_keys(['4912208', '4912223', '9909282', '4912205', '4912209'])

In [7]:
diccionario_de_sondas_en_fechas = carga.seleccionar_rango_de_fechas(diccionario = diccionario_de_datos_de_sondas, buscar_fechas_anteriores_al_estudio = False, config = config)

c:\programacion\codigos_python\drift_buoys\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [8]:
datos_de_sondas_sin_duplicados = carga.buscar_y_eliminar_duplicados(diccionario_de_sondas_en_fechas)

In [9]:
datos_ordenados = carga.ordernar_datos_por_fecha(diccionario_de_sondas_en_fechas)

Datos ordenados por fecha para la sonda 4912208.
Datos ordenados por fecha para la sonda 4912223.
Datos ordenados por fecha para la sonda 9909282.
Datos ordenados por fecha para la sonda 4912205.
Datos ordenados por fecha para la sonda 4912209.


In [10]:
datos_ordenados.keys()

dict_keys(['4912208', '4912223', '9909282', '4912205', '4912209'])

In [11]:
for serial in seriales_encontrados:
    if serial in datos_ordenados:
        datos_ordenados[serial]["tspan_rounded"] = datos_ordenados[serial]["tspan_de_envio"]


In [12]:
# Eliminar datos espurios (solo se revisa si hay valores de rapidez superiores a 2 m/s y se elimina toda la fila)
datos_finales = eliminar_datos_espurios(datos_ordenados)

In [13]:
# Agregar componentes de la velocidad al diccionario con los dataframe de cada sonda
datos_finales = carga.agregar_componentes_de_la_velocidad(datos_finales)

In [14]:
datos_finales["4912208"].columns

Index(['tspan_de_envio', 'latitud', 'longitud', 'rap_corriente', 'distancia',
       'dir_corriente', 'dir_corriente_texto', 'temperatura_mar', 'voltaje',
       'tspan_rounded', 'u_corriente', 'v_corriente'],
      dtype='str')

In [15]:
# Crear ruta a la carpeta de guardado de datos de laboratorio
ruta_a_carpeta = config.carpeta_de_guardado_de_datos_procesados 
fecha_del_estudio = config.convertir_a_pd_datetime("fecha_del_estudio", formato="%Y-%m-%d")
carpeta_del_estudio = f"{fecha_del_estudio.year:04d}{fecha_del_estudio.month:02d}"
ruta_a_la_carpeta_de_guardado = os.path.join(ruta_a_carpeta, carpeta_del_estudio)

In [16]:
ruta_a_la_carpeta_de_guardado

'C:\\Users\\Atmosfera\\Desktop\\datos_procesados\\doris\\10.3\\202607'

In [17]:
# Guardar datos del estudio en archivo .pkl
carpeta_de_destino = ruta_a_la_carpeta_de_guardado
nombre_de_archivo = config.nombre_del_archivo_de_datos_procesados
guardar_diccionario_como_pickle(diccionario = datos_finales, 
                                ruta = carpeta_de_destino, 
                                nombre_archivo=nombre_de_archivo)


Diccionario guardado correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\datos_procesados_sondas_oceanograficas.pkl


In [18]:
tabla_de_porcentajes = calcular_porcentaje_de_datos_recibidos(datos_finales, config)

c:\programacion\codigos_python\drift_buoys\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [19]:
tabla_de_porcentajes

,serial_de_sonda,fecha_de_inicio,fecha_final,cantidad_de_datos_esperados,cantidad_de_datos_recibidos,porcentaje_de_datos_recibidos
0,4912208,2026-07-20 21:53:00,2026-07-31 23:18:00,532,478,89.85
1,4912223,2026-07-20 22:27:00,2026-07-31 23:42:00,532,513,96.43
2,9909282,2026-07-20 22:45:00,2026-07-31 23:56:00,531,523,98.49
3,4912205,2026-07-20 23:16:00,2026-07-31 23:58:00,530,517,97.55
4,4912209,2026-07-20 23:13:00,2026-07-31 23:56:00,530,512,96.60


In [20]:
# Guardar Tabla de porcentajes
guardar_porcentajes_en_excel(data= tabla_de_porcentajes, ruta= ruta_a_la_carpeta_de_guardado, nombre_de_archivo=config.nombre_del_excel_de_porcentajes)

Archivo Excel guardado correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\porcentajes_de_las_sondas.xlsx


In [21]:
graficar_series_y_guardar(
    datos= datos_finales,
    ruta_a_carpeta_de_guardado=ruta_a_la_carpeta_de_guardado, 
    mostrar_figura=False, 
    config=config
)

Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\transmision_4912208.png
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\transmision_4912223.png
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\transmision_9909282.png
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\transmision_4912205.png
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\transmision_4912209.png


In [22]:
graficar_mapa_de_posiciones(
    datos=datos_finales, 
    ruta_a_la_carpeta_de_guardado = ruta_a_la_carpeta_de_guardado,
    mostrar_figura=False,
    config=config)

c:\programacion\codigos_python\drift_buoys\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


sonda 4912208 lat ini= 19.362062, lon ini= -92.01112
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\mapa_trayectoria_4912208.png
sonda 4912223 lat ini= 19.361472, lon ini= -92.04998
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\mapa_trayectoria_4912223.png
sonda 9909282 lat ini= 19.360582, lon ini= -92.030153
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\mapa_trayectoria_9909282.png
sonda 4912205 lat ini= 19.396223, lon ini= -92.044659
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\mapa_trayectoria_4912205.png
sonda 4912209 lat ini= 19.396191, lon ini= -92.023458
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\mapa_trayectoria_4912209.png
